In [ ]:
import os  # Import the operating system module for environment configurations
import json  # Import JSON module to read/write system configuration and telemetry files
import time  # Import time module to handle benchmarks, tracking, and timeouts
import cv2  # Import OpenCV library for advanced image and video processing tasks
import torch  # Import PyTorch deep learning framework for GPU compute allocation
import numpy as np  # Import NumPy for fast numerical and matrix operations on arrays
from threading import Thread  # Import Thread to run the stream reader in a parallel background process
from ultralytics import YOLO  # Import YOLO from ultralytics to load and run object detection models

# Prevent initialization crashes related to duplicate OpenMP libraries inside the notebook runtime
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# Evaluate hardware ecosystem to check if a compatible Nvidia CUDA GPU is available on the local machine
if torch.cuda.is_available():
    torch.cuda.set_device(0)  # Bind execution explicitly to the first available graphics card index
    torch.cuda.empty_cache()  # Clear unused allocated memory from the GPU to optimize dynamic VRAM spaces
    DEVICE_IDX = 0  # Store the active GPU index integer for references inside the model engine
    DEVICE_NAME = torch.cuda.get_device_name(0)  # Retrieve the human-readable hardware descriptor name of the GPU
else:
    DEVICE_IDX = "cpu"  # Fallback to standard CPU processing index if no Nvidia graphics card is detected
    DEVICE_NAME = "CPU"  # Set hardware descriptor name to CPU for pipeline logging clarity

CONFIG_PATH = "config.json"  # Define the standard local text file path for reading user boundary configurations
TELEMETRY_PATH = "telemetry.json"  # Define the output file path where counting analytics statistics are saved
UDP_URL = "udp://0.0.0.0:9999?listen&timeout=2000000"  # Set up connection socket string with a 2-second timeout parameter

print(f"[INFO] Initial hardware acceleration environment configured successfully on device: {DEVICE_NAME}")

In [ ]:
class RTSPVideoStream:  # Define a dedicated class to handle non-blocking real-time video streaming over network protocols
    def __init__(self, src):  # Initialize the stream reader class constructor with the source UDP endpoint URL
        self.stream = cv2.VideoCapture(src, cv2.CAP_FFMPEG)  # Open the video stream using FFMPEG decoding backend wrapper
        self.stream.set(cv2.CAP_PROP_BUFFERSIZE, 1)  # Force internal hardware buffer stack to minimum size of 1 frame
        self.stopped = False  # Set boolean control flag to manage the background worker thread execution life cycle
        self.ret = False  # Initialize the frame retrieval success status indicator flag to structural False baseline
        self.frame = None  # Initialize frame variable storage container to empty state holds no image array matrix yet

    def start(self):  # Define start function to kick off parallel background thread execution routines
        t = Thread(target=self.update, args=())  # Instantiate a separate parallel worker thread targeting frame update loop
        t.daemon = True  # Mark thread as background daemon so it exits automatically when main notebook kernel terminates
        t.start()  # Launch the thread execution path to begin draining and decoding network socket packets immediately
        return self  # Return instance object reference to allow chained command method execution patterns down the line

    def update(self):  # Define background execution loop that constantly drains data layers from the network socket connection
        while True:  # Enter infinite loop structure to constantly listen for incoming video transmission packets
            if self.stopped:  # Evaluate if external program instruction sequence has requested thread termination routines
                break  # Break out of execution loop immediately to safely close worker process down without leaks
            self.ret, self.frame = self.stream.read()  # Grab and fully decode next sequential stream packet immediately into memory
            if not self.ret:  # Check if stream transmission failed or returned empty data structure packet assets
                self.stopped = True  # Toggle shutdown flag state to stop operations and initiate reconnection routines

    def read(self):  # Define public access function to return latest frame cache registry data instantly
        return self.ret, self.frame  # Return verification status boolean together with the most recent frame matrix layer

    def stop(self):  # Define stop function to handle clean teardown routines on hardware network interfaces
        self.stopped = True  # Toggle thread control flag state to terminate active background update loops instantly
        self.stream.release()  # Close network socket bindings and clear active system video capture handler channels

def load_dashboard_config():  # Declare a function to load updated line and polygon zone metrics directly from disk
    if os.path.exists(CONFIG_PATH):  # Check if the configuration JSON file is present on the local storage drive system
        try:  # Initialize a protective try-catch block to handle syntax errors or file access permission bugs
            with open(CONFIG_PATH, "r") as f:  # Open the configuration file in safe read-only text file mode
                return json.load(f)  # Parse the text file content into a native functional Python dictionary structure
        except Exception:  # Intercept any parsing errors to ensure pipeline operations do not crash unexpectedly
            pass  # Ignore the file error seamlessly and fall back to the default fallback configuration template below
    return {"counting_line_ratio": 0.50, "zones": []}  # Return default values if configuration file missing or corrupted

def write_telemetry(total_in, total_out, inside, fps):  # Declare a function to output current analytics metadata
    data = {  # Construct a dictionary containing all metrics updated in the current frame iteration loop
        "total_in": total_in,  # Map the accumulated inward crossing counter to the output payload data structure
        "total_out": total_out,  # Map the accumulated outward crossing counter to the output payload data structure
        "current_inside": inside,  # Map the count of unique objects within the designated polygon boundary lines
        "fps": round(fps, 1)  # Map the computed system processing speed rounded to one decimal place value
    }  # End of data payload dictionary instantiation pattern
    try:  # Open a safe execution block to prevent storage drive access failures from interrupting the system
        with open(TELEMETRY_PATH, "w") as f:  # Open the designated telemetry output file in write mode
            json.dump(data, f)  # Serialize and write the metrics dictionary as clean formatted json text data
    except Exception:  # Intercept file lock or disk capacity exceptions safely without freezing execution
        pass  # Bypass tracking file errors gracefully to prioritize live video stream rendering execution speed

print("[SUCCESS] Background core stream classes and telemetry helper utilities compiled successfully.")

In [ ]:
print(f"[INFO] Loading YOLO model weights onto hardware layer: {DEVICE_NAME}...")

# Load the neural network architecture weights and force memory allocation onto the designated hardware accelerator device index
model = YOLO("yolo26l.pt", task="detect").to(f"cuda:{DEVICE_IDX}" if torch.cuda.is_available() else "cpu")

print("[SUCCESS] YOLO Model weights successfully initialized, allocated, and cached in system memory!")

In [ ]:
def process_counting_logic(frame, h, w, config, boxes, ids, clses, track_history, crossed, entry_times, time_stats):
    """
    This function isolates all drawing and line-crossing mathematical logic parameters.
    It now includes time tracking and color-coded bounding boxes based on state and class.
    """
    global total_in, total_out  # Bring global counter variables into function scope
    
    # ----------------------------------------------------------------------------------------------------
    # LINE ORIENTATION & POSITION SETUP 
    # ----------------------------------------------------------------------------------------------------
    line_ratio = config.get("counting_line_ratio", 0.50)  
    line_pos_x = int(w * line_ratio)  
    cv2.line(frame, (line_pos_x, 0), (line_pos_x, h), (0, 0, 255), 3)  

    current_time = time.time() # Capture the current timestamp for tracking duration

    # ----------------------------------------------------------------------------------------------------
    # INDIVIDUAL OBJECT TRACKING AND TIME MATHEMATICS LOOP
    # ----------------------------------------------------------------------------------------------------
    for box, tid, cls in zip(boxes, ids, clses):  # Extract box coordinates, ID, and class
        cx = int((box[0] + box[2]) / 2)  
        cy = int((box[1] + box[3]) / 2)  

        # --- TIME TRACKING LOGIC ---
        if tid not in entry_times:
            entry_times[tid] = current_time # Record the exact timestamp when the ID first appears
            
        # Calculate how many seconds this specific ID has been on screen
        time_spent = current_time - entry_times[tid]
        time_stats[tid] = time_spent # Store the duration in the stats dictionary

        # --- DIRECTION COUNTING LOGIC ---
        prev_cx = track_history.get(tid, cx)  
        
        if prev_cx <= line_pos_x and cx > line_pos_x and crossed.get(tid) != "IN":  
            total_in += 1  
            crossed[tid] = "IN"  
        elif prev_cx >= line_pos_x and cx < line_pos_x and crossed.get(tid) != "OUT":  
            total_out += 1  
            crossed[tid] = "OUT"  

        track_history[tid] = cx  

        # --- VISUAL BOUNDING BOX DRAWING (COLOR CODED) ---
        # Assuming YOLO is trained so Class 0 = Customer, Class 1 = Employee
        if cls == 1:
            color = (255, 0, 255) # Purple for Employees
            status_text = "EMP"
        elif crossed.get(tid) == "IN":
            color = (0, 255, 0) # Green for IN
            status_text = "IN"
        elif crossed.get(tid) == "OUT":
            color = (0, 0, 255) # Red for OUT
            status_text = "OUT"
        else:
            color = (255, 255, 0) # Cyan for Default/Tracking before crossing
            status_text = "TRACK"

        # Draw the colored bounding box
        cv2.rectangle(frame, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), color, 2)  
        
        # Display ID, Status, and Time Spent
        display_label = f"{status_text} #{tid} | {int(time_spent)}s"
        cv2.putText(frame, display_label, (int(box[0]), int(box[1]) - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)  

    print("[SUCCESS] Counting, timing, and colored drawing logic updated successfully!")

In [ ]:
# Reset runtime counting states
total_in = 0  
total_out = 0  
track_history = {}  
crossed = {}  
frame_count = 0  
entry_times = {}  # Dictionary to store the initial entry timestamp for each unique ID
time_stats = {}   # Dictionary to store the current accumulated time for each unique ID

window_name = "AETHER VISION v5.0 - MODULAR IPYNB CORE"  
cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)  

user_requested_exit = False  

while True:  
    if user_requested_exit:  
        break  
        
    print(f"[INFO] Connecting to pipeline feed thread instance: {UDP_URL}")  
    vstream = RTSPVideoStream(UDP_URL).start()  
    time.sleep(1.0)  

    if not vstream.ret:  
        print("[ERROR] Core connection failed to parse packets. Retrying in 3 seconds...")  
        vstream.stop()  
        time.sleep(3)  
        continue  

    print("[SUCCESS] Connected! Streaming frame buffer live. Press 'q' to stop.")  
    config = load_dashboard_config()  
    last_frame_time = time.time()  

    while True:  
        t0 = time.time()  
        frame_count += 1  

        if frame_count % 10 == 0:  
            config = load_dashboard_config()  

        ret, frame = vstream.read()  
        if not ret or frame is None:  
            if time.time() - last_frame_time > 3.0:  
                print("[WARNING] Connection timeout. Retrying network link...")  
                vstream.stop()  
                break  
            time.sleep(0.001)  
            continue  

        last_frame_time = time.time()  
        h, w, _ = frame.shape  
        
        # NOTE: If you haven't trained a model for employees, change classes=[0] to classes=[0, 1] once you do
        results = model.track(frame, imgsz=640, conf=0.25, classes=[0], tracker="bytetrack.yaml", persist=True, verbose=False, device=DEVICE_IDX)

        current_detected_count = 0  
        
        if results[0].boxes.id is not None:  
            boxes = results[0].boxes.xyxy.cpu().numpy()  
            ids = results[0].boxes.id.cpu().numpy().astype(int)  
            clses = results[0].boxes.cls.cpu().numpy().astype(int) # Extract class IDs to identify employees

            current_detected_count = len(ids)  
            
            # Pass the new class arrays and time dictionaries to the logic function
            _ = process_counting_logic(frame, h, w, config, boxes, ids, clses, track_history, crossed, entry_times, time_stats)

        # Memory Cleanup (Ensure we also remove old IDs from time tracking dictionaries)
        if len(track_history) > 300:  
            excess_keys = list(track_history.keys())[:-100]  
            for k in excess_keys:  
                track_history.pop(k, None)  
                crossed.pop(k, None)  
                entry_times.pop(k, None) # Free memory from time caches
                time_stats.pop(k, None)

        net_inside_count = max(0, total_in - total_out)  
        fps = 1.0 / max(time.time() - t0, 0.001)  
        
        write_telemetry(total_in, total_out, net_inside_count, fps)  

        # --- CALCULATE AVERAGE TIME IN SCREEN ---
        active_times = []
        if results[0].boxes.id is not None:
            # Gather the time spent for all IDs currently visible in this frame
            active_times = [time_stats[tid] for tid in ids if tid in time_stats]
        
        # Compute the mean average time if there are people on screen, otherwise default to 0
        avg_time = sum(active_times) / len(active_times) if active_times else 0.0

        # UI Overlay Construction
        overlay = frame.copy()  
        cv2.rectangle(overlay, (0, 0), (w, 55), (20, 20, 20), -1)  
        cv2.addWeighted(overlay, 0.75, frame, 0.25, 0, frame)  

        # Format header text with the new Average Time metric
        header_text = f"IN: {total_in} | OUT: {total_out} | INSIDE: {net_inside_count} | AVG TIME: {int(avg_time)}s | {round(fps, 1)} FPS"
        cv2.putText(frame, header_text, (20, 36), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)  

        cv2.imshow(window_name, frame)  
        
        if cv2.waitKey(1) & 0xFF == ord('q'):  
            print("[INFO] 'q' key pressed. Shutting down...")  
            vstream.stop()  
            cv2.destroyAllWindows()  
            user_requested_exit = True  
            break
        # --- END OF FRAME PROCESSING LOOP ---  